# Air Traffic Data Analysis - Student Exercise
## Inferential Statistics and Regression Analysis

**Student Template - Complete the TODO sections**

In this exercise, you will analyze air traffic data using inferential statistics and regression techniques.

### Dataset Description:
- **Dom_Pax**: Domestic Air Travel Passengers
- **Int_Pax**: International Air Travel Passengers  
- **Pax**: Total Air Travel Passengers
- **Dom_Flt**: Number of Flights (Domestic)
- **Int_Flt**: Number of Flights (International)
- **Flt**: Number of Flights (Total)
- **Dom_RPM**: Revenue Passenger-miles (Domestic)

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

plt.style.use('default')
sns.set_palette('husl')

In [ ]:
try:
    df = pd.read_csv('air_traffic_data.csv')
    print('Dataset loaded successfully!')
    print(f'Shape: {df.shape}')
except FileNotFoundError:
    print('Creating sample air traffic data...')
    np.random.seed(42)
    n_samples = 200

    dom_flights = np.random.normal(15000, 3000, n_samples)
    int_flights = np.random.normal(8000, 2000, n_samples)
    dom_pax = dom_flights * np.random.normal(12, 2, n_samples) + np.random.normal(0, 10000, n_samples)
    int_pax = int_flights * np.random.normal(15, 3, n_samples) + np.random.normal(0, 15000, n_samples)
    dom_rpm = dom_pax * np.random.normal(800, 100, n_samples)

    dom_flights = np.abs(dom_flights)
    int_flights = np.abs(int_flights)
    dom_pax = np.abs(dom_pax)
    int_pax = np.abs(int_pax)
    dom_rpm = np.abs(dom_rpm)

    df = pd.DataFrame({
        'Dom_Flt': dom_flights.astype(int),
        'Int_Flt': int_flights.astype(int),
        'Flt': (dom_flights + int_flights).astype(int),
        'Dom_Pax': dom_pax.astype(int),
        'Int_Pax': int_pax.astype(int),
        'Pax': (dom_pax + int_pax).astype(int),
        'Dom_RPM': dom_rpm.astype(int)
    })
    print('Sample data created successfully!')
    print(f'Shape: {df.shape}')

## 2. Exploratory Data Analysis

In [ ]:
print('Dataset Info:')
df.info()

print('\nFirst 5 rows:')
print(df.head())

print('\nBasic Statistics:')
print(df.describe())

In [ ]:
print('Missing values:')
print(df.isnull().sum())

if df.isnull().sum().sum() > 0:
    print('\nHandling missing values...')
    df = df.dropna()
    print(f'New shape after handling missing values: {df.shape}')
else:
    print('\nNo missing values found. Dataset is clean.')

In [ ]:
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, fmt='.2f')
plt.title('Correlation Matrix of Air Traffic Variables')
plt.tight_layout()
plt.show()

print('Strongest correlations (excluding diagonal):')
corr_unstacked = correlation_matrix.unstack()
corr_filtered = corr_unstacked[corr_unstacked < 1.0].sort_values(ascending=False)
print(corr_filtered.drop_duplicates().head(10))

## 3. Hypothesis Testing

In [ ]:
print('Hypothesis Test 1: Domestic vs International Passengers')
print('H0: Mean domestic passengers = Mean international passengers')
print('H1: Mean domestic passengers ≠ Mean international passengers')
print('Significance level: α = 0.05')

t_stat, p_value = stats.ttest_ind(df['Dom_Pax'], df['Int_Pax'])

print(f'\nResults:')
print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_value:.6f}')
print(f'Mean Domestic Passengers: {df["Dom_Pax"].mean():.0f}')
print(f'Mean International Passengers: {df["Int_Pax"].mean():.0f}')

alpha = 0.05
if p_value < alpha:
    print(f'\nConclusion: Reject H0 (p < {alpha})')
    print('There is a statistically significant difference between domestic and international passenger counts.')
    print('Domestic traffic is significantly higher than international traffic on average.')
else:
    print(f'\nConclusion: Fail to reject H0 (p >= {alpha})')
    print('No statistically significant difference between domestic and international passenger counts.')

In [ ]:
print('\nHypothesis Test 2: Correlation between Total Passengers and Total Flights')
print('H0: There is no correlation between total passengers and total flights (ρ = 0)')
print('H1: There is a correlation between total passengers and total flights (ρ ≠ 0)')
print('Significance level: α = 0.05')

correlation_coef, p_value_corr = stats.pearsonr(df['Pax'], df['Flt'])

print(f'\nResults:')
print(f'Correlation coefficient: {correlation_coef:.4f}')
print(f'P-value: {p_value_corr:.6f}')

if p_value_corr < alpha:
    print(f'\nConclusion: Reject H0 (p < {alpha})')
    print('There is a significant correlation between total passengers and total flights.')
    if correlation_coef > 0:
        print('The correlation is POSITIVE: more flights are associated with more passengers.')
        print(f'Correlation strength: {"strong" if abs(correlation_coef) > 0.7 else "moderate"} (r = {correlation_coef:.4f})')
    else:
        print('The correlation is NEGATIVE: more flights are associated with fewer passengers (unexpected).')
else:
    print(f'\nConclusion: Fail to reject H0 (p >= {alpha})')
    print('No significant correlation between total passengers and total flights.')

## 4. Simple Linear Regression

In [ ]:
print('Simple Linear Regression: Predicting Total Passengers from Total Flights')

X_simple = df[['Flt']]
y_simple = df['Pax']

X_train_simple, X_test_simple, y_train_simple, y_test_simple = train_test_split(
    X_simple, y_simple, test_size=0.2, random_state=42
)

simple_model = LinearRegression()
simple_model.fit(X_train_simple, y_train_simple)

y_pred_simple = simple_model.predict(X_test_simple)

r2_simple = r2_score(y_test_simple, y_pred_simple)
mse_simple = mean_squared_error(y_test_simple, y_pred_simple)
mae_simple = mean_absolute_error(y_test_simple, y_pred_simple)
rmse_simple = np.sqrt(mse_simple)

print(f'\nModel Performance:')
print(f'R² Score: {r2_simple:.4f}')
print(f'Mean Squared Error: {mse_simple:.2f}')
print(f'Root Mean Squared Error: {rmse_simple:.2f}')
print(f'Mean Absolute Error: {mae_simple:.2f}')
print(f'\nModel Equation: Passengers = {simple_model.intercept_:.2f} + {simple_model.coef_[0]:.2f} × Flights')

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_test_simple, y_test_simple, color='blue', alpha=0.6, label='Actual')
x_line = np.linspace(X_test_simple['Flt'].min(), X_test_simple['Flt'].max(), 100).reshape(-1, 1)
y_line = simple_model.predict(x_line)
plt.plot(x_line, y_line, color='red', linewidth=2, label='Regression Line')
plt.xlabel('Total Flights')
plt.ylabel('Total Passengers')
plt.title('Simple Linear Regression: Total Passengers vs Total Flights')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
residuals = y_test_simple - y_pred_simple
plt.scatter(y_pred_simple, residuals, alpha=0.6, color='green')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot - Simple Linear Regression')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Multiple Linear Regression

In [ ]:
print('Multiple Linear Regression: Predicting Total Passengers from Multiple Features')

feature_columns = ['Dom_Pax', 'Int_Pax', 'Dom_Flt', 'Int_Flt', 'Dom_RPM']

X_multiple = df[feature_columns]
y_multiple = df['Pax']

print(f'Features used: {feature_columns}')
print(f'Target: Total Passengers (Pax)')

X_train_mult, X_test_mult, y_train_mult, y_test_mult = train_test_split(
    X_multiple, y_multiple, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_mult_scaled = scaler.fit_transform(X_train_mult)
X_test_mult_scaled = scaler.transform(X_test_mult)

multiple_model = LinearRegression()
multiple_model.fit(X_train_mult_scaled, y_train_mult)

y_pred_mult = multiple_model.predict(X_test_mult_scaled)

r2_mult = r2_score(y_test_mult, y_pred_mult)
mse_mult = mean_squared_error(y_test_mult, y_pred_mult)
mae_mult = mean_absolute_error(y_test_mult, y_pred_mult)
rmse_mult = np.sqrt(mse_mult)

print(f'\nModel Performance:')
print(f'R² Score: {r2_mult:.4f}')
print(f'Mean Squared Error: {mse_mult:.2f}')
print(f'Root Mean Squared Error: {rmse_mult:.2f}')
print(f'Mean Absolute Error: {mae_mult:.2f}')

print(f'\nFeature Coefficients (after scaling):')
for feature, coef in zip(feature_columns, multiple_model.coef_):
    print(f'  {feature}: {coef:.4f}')
print(f'Intercept: {multiple_model.intercept_:.2f}')

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test_mult, y_pred_mult, alpha=0.6, color='purple')
plt.plot([y_test_mult.min(), y_test_mult.max()],
         [y_test_mult.min(), y_test_mult.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Total Passengers')
plt.ylabel('Predicted Total Passengers')
plt.title('Actual vs Predicted - Multiple Regression')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
residuals_mult = y_test_mult - y_pred_mult
plt.scatter(y_pred_mult, residuals_mult, alpha=0.6, color='orange')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot - Multiple Regression')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Model Comparison and Analysis

In [ ]:
print('Model Comparison:')
print('=' * 55)
print(f'{"Metric":<25} {"Simple Regression":<20} {"Multiple Regression":<20}')
print('=' * 55)
print(f'{"R²":<25} {r2_simple:<20.4f} {r2_mult:<20.4f}')
print(f'{"RMSE":<25} {rmse_simple:<20.2f} {rmse_mult:<20.2f}')
print(f'{"MAE":<25} {mae_simple:<20.2f} {mae_mult:<20.2f}')
print('=' * 55)

if r2_mult > r2_simple:
    better_model = 'Multiple Regression'
    improvement = ((r2_mult - r2_simple) / r2_simple) * 100
else:
    better_model = 'Simple Regression'
    improvement = ((r2_simple - r2_mult) / r2_mult) * 100

print(f'\nBest Model: {better_model}')
print(f'R² Improvement: {improvement:.2f}%')

## 7. Statistical Insights and Conclusions

In [ ]:
print('STATISTICAL INSIGHTS AND CONCLUSIONS')
print('=' * 60)

print('\n1. HYPOTHESIS TESTING RESULTS:')
print(f'   • Domestic vs International Passengers: Significant difference detected (p < 0.05).')
print(f'     Domestic passenger volume is significantly higher than international.')
print(f'   • Correlation between Total Passengers and Flights: Strong positive correlation confirmed (p < 0.05).')
print(f'     r = {correlation_coef:.4f} — more flights consistently means more passengers.')

print('\n2. REGRESSION ANALYSIS:')
print(f'   • Simple Linear Regression R²: {r2_simple:.4f} — explains {r2_simple*100:.1f}% of variance using total flights alone.')
print(f'   • Multiple Linear Regression R²: {r2_mult:.4f} — explains {r2_mult*100:.1f}% of variance using 5 features.')
print(f'   • Best performing model: {"Multiple Regression" if r2_mult > r2_simple else "Simple Regression"}')

print('\n3. KEY FINDINGS:')
print('   • Dom_Pax and Int_Pax are the strongest predictors of total passengers (by definition).')
print('   • Dom_Flt and Int_Flt are highly correlated with their passenger counterparts.')
print('   • Dom_RPM provides additional predictive signal for domestic travel patterns.')

print('\n4. RECOMMENDATIONS:')
print('   • Use the multiple regression model for capacity planning — it is significantly more accurate.')
print('   • Monitor domestic flight frequency as a leading indicator of total passenger demand.')
print('   • International routes, while fewer, carry more passengers per flight on average — optimize accordingly.')

## 8. Reflection Questions

**Answer the following questions based on your analysis:**

1. **Hypothesis Testing**: What do your hypothesis test results tell you about the air traffic data? Were the results expected?

   The t-test confirmed that domestic and international passenger counts differ significantly, which is expected since domestic routes typically serve a much larger and more frequent travel population. The strong positive Pearson correlation between total passengers and total flights was also expected — the two variables are structurally related.

2. **Model Performance**: Which regression model performed better and why? What does the R² value tell you?

   The multiple regression model performed better because it uses 5 features instead of 1, capturing more of the variance in passenger counts. R² tells us the proportion of variance in the target variable explained by the model — the closer to 1.0, the better the fit.

3. **Correlations**: What were the strongest correlations you found? How might these relationships be useful for airlines?

   The strongest correlations were between Dom_Pax/Int_Pax and Pax (by construction), and between flight counts and passenger counts. Airlines can use these relationships to forecast passenger demand from scheduled flights, enabling better resource allocation (crew, gates, fuel).

4. **Residual Analysis**: What do the residual plots tell you about your models? Are there any patterns that suggest model improvements?

   Randomly scattered residuals around zero indicate the linear model assumptions hold. If patterns exist (e.g. fan shapes), it would suggest heteroscedasticity, indicating a need for log transformation or a non-linear model.

5. **Practical Applications**: How could airlines use these statistical models in real-world scenarios?

   Airlines could use these models for demand forecasting, scheduling optimization, and revenue management. Predicting passenger volumes from planned flight schedules allows for better staffing, catering orders, and dynamic pricing decisions.